# TP3 — xG desde cero (informe)

**Nombre:** Matías · **Entrega A:** _fecha_ · **Entrega B:** _fecha_

Usa `mv.prob` y `mv.xg`. `sklearn` aparece **solo para comparar**. Respondé en las celdas **✍️ Respuesta**.

Antes: `python herramientas/bajar_tiros_statsbomb.py`.
Datos: StatsBomb Open Data (uso no comercial, con atribución).

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mv import prob as pr
from mv import xg
from mv.datos import RAIZ

tiros = pd.read_csv(RAIZ / "datos" / "statsbomb" / "tiros.csv")
print(tiros.shape); tiros.head()

In [ ]:
def dibujar_media_cancha(ax=None):
    """Media cancha StatsBomb (120 × 80 yd): solo líneas, el arco a la derecha."""
    ax = ax or plt.gca()
    kw = dict(color="0.4", lw=1)
    ax.plot([60, 120, 120, 60, 60], [0, 0, 80, 80, 0], **kw)                  # media cancha
    ax.plot([120, 102, 102, 120], [18, 18, 62, 62], **kw)                    # área grande
    ax.plot([120, 114, 114, 120], [30, 30, 50, 50], **kw)                    # área chica
    ax.plot([120, 120], [36, 44], color="k", lw=4)                           # arco
    ax.plot(108, 40, "o", color="0.4", ms=3)                                 # penal
    ax.set_xlim(58, 122); ax.set_ylim(82, -2); ax.set_aspect("equal"); ax.axis("off")
    return ax

---
# Parte A — Probabilidad

## A1. Distribución de goles: exacta contra Monte Carlo

In [ ]:
xgs = [0.1, 0.3, 0.5, 0.05, 0.22, 0.08]
exacta = pr.distribucion_goles(xgs)
sims = pr.simular_goles(xgs, 50_000, np.random.default_rng(0))
frec = np.bincount(sims, minlength=len(exacta)) / len(sims)

k = np.arange(len(exacta))
plt.bar(k - 0.2, exacta, width=0.4, label="exacta (convolución)")
plt.bar(k + 0.2, frec, width=0.4, label="Monte Carlo (50.000)")
plt.xlabel("goles"); plt.ylabel("probabilidad"); plt.legend(); plt.show()
print(f"E[goles] = {(k * exacta).sum():.3f} (suma de xG = {sum(xgs):.3f}) | P(al menos uno) = {pr.prob_al_menos_uno(xgs):.3f}")

In [ ]:
# TODO: ¿cómo baja el error de Monte Carlo con n? Para n = 100, 1000, 10_000, 100_000:
# repetí 50 veces la simulación con distintas semillas y medí el desvío de la estimación de P(0 goles).
# Graficá desvío vs n en escala log-log. ¿La pendiente es −1/2?

**Preguntas**
1. ¿Coinciden la distribución exacta y la simulada? ¿Qué pendiente te dio el gráfico log-log, y por qué es esa?
2. Con estos 6 tiros, ¿cuál es el resultado más probable? ¿Coincide con "redondear el xG"?

✍️ **Respuesta:**

## A2. Bayes: el detector y la tasa base

In [ ]:
tasas = np.logspace(-3, np.log10(0.5), 100)
post = [pr.bayes(t, 0.9, 0.05) for t in tasas]
plt.semilogx(tasas, post); plt.axhline(0.5, color="0.6", ls="--")
plt.xlabel("tasa base: P(hay pelota)"); plt.ylabel("P(hay pelota | el detector dice pelota)")
plt.title("detector con 90 % de recall y 5 % de falsas alarmas"); plt.grid(alpha=.3); plt.show()

**Preguntas**
3. ¿Desde qué tasa base más de la mitad de las alarmas son falsas? Si buscás la pelota en recortes chicos de un frame 1080p, ¿en qué zona del gráfico estás?
4. Para mejorar la precisión con una tasa base baja, ¿conviene subir el recall o bajar las falsas alarmas? Mostralo con números.

✍️ **Respuesta:**

## A3. ¿Dónde se tira? Una gaussiana sobre las posiciones

In [ ]:
op = tiros[tiros.tipo == "Open Play"]
P = op[["x", "y"]].to_numpy()
mu = P.mean(axis=0)
Sigma = (P - mu).T @ (P - mu) / len(P)
print("μ =", mu.round(2)); print("Σ =\n", Sigma.round(2))

ax = dibujar_media_cancha()
ax.scatter(P[:, 0], P[:, 1], s=2, alpha=.2)
for k_, col in ((1, "tab:orange"), (2, "tab:red")):
    E = pr.elipse_confianza(mu, Sigma, k=k_)
    ax.plot(E[:, 0], E[:, 1], color=col, lw=2, label=f"{k_} desvío(s) (Mahalanobis)")
ax.legend(loc="lower left"); plt.show()

d = pr.mahalanobis(P, mu, Sigma)
print(f"fracción dentro de 1 desvío: {(d <= 1).mean():.2f} (una gaussiana 2D: 0,39) | dentro de 2: {(d <= 2).mean():.2f} (0,86)")

**Preguntas**
5. Interpretá Σ: ¿en qué dirección se dispersan más los tiros? ¿Hay correlación entre x e y?
6. Las fracciones dentro de 1 y 2 desvíos, ¿se parecen a las de una gaussiana 2D (0,39 y 0,86)? ¿Por qué una gaussiana **no** es un buen modelo para "dónde se tira"? (Mirá la línea de fondo.)

✍️ **Respuesta:**

---
# Parte B — El modelo de xG

## B1. Features y split por partido

In [ ]:
op = tiros[tiros.tipo == "Open Play"].reset_index(drop=True)
dist, ang = xg.distancia_y_angulo(op.x.to_numpy(), op.y.to_numpy())
cabeza = (op.parte_cuerpo == "cabeza").to_numpy().astype(float)
y = op.gol.to_numpy()

test = xg.split_por_partido(op.match_id.to_numpy(), 0.2, np.random.default_rng(0))
print(f"train: {(~test).sum()} tiros, {y[~test].mean():.3f} de goles | test: {test.sum()} tiros, {y[test].mean():.3f}")

## B2. Solo distancia, y el *learning rate*

In [ ]:
X1 = dist[:, None]
X1_tr, m1, s1 = xg.estandarizar(X1[~test])
X1_te, _, _ = xg.estandarizar(X1[test], m1, s1)

plt.figure(figsize=(9, 4))
for lr in (0.01, 0.1, 1.0, 5.0):
    w, b, hist = xg.entrenar_logistica(X1_tr, y[~test], lr=lr, n_iter=500)
    plt.plot(hist, label=f"η = {lr}")
plt.xlabel("iteración"); plt.ylabel("log loss (train)"); plt.legend(); plt.ylim(0.25, 0.7); plt.show()

**Preguntas**
7. ¿Qué η converge más rápido? ¿Alguno oscila o diverge? Probá también η = 50 y contá qué pasa.
8. ¿Por qué con datos **sin** estandarizar (probalo: `dist[:, None]` directo) hace falta un η mucho más chico?

✍️ **Respuesta:**

## B3. Distancia + ángulo + cabeza

In [ ]:
X = np.column_stack([dist, ang, cabeza])
X_tr, m, s = xg.estandarizar(X[~test])
X_te, _, _ = xg.estandarizar(X[test], m, s)
w, b, hist = xg.entrenar_logistica(X_tr, y[~test], lr=0.5, n_iter=3000)
print("w =", w.round(3), " b =", round(b, 3))

from sklearn.linear_model import LogisticRegression
ref = LogisticRegression(C=1e8, max_iter=10_000).fit(X_tr, y[~test])
print("sklearn:", ref.coef_[0].round(3), round(ref.intercept_[0], 3))

In [ ]:
# Interpretación: cuánto se multiplican los odds de gol por cada "unidad" de cada feature.
# Ojo: X está estandarizada → una unidad = un DESVÍO de esa feature. Pasalo a unidades reales.
for nombre, wj, sj in zip(["distancia (yd)", "ángulo (rad)", "cabeza (0/1)"], w, s):
    print(f"{nombre:15s}  odds × {np.exp(wj):.2f} por desvío ({sj:.2f})   →  × {np.exp(wj / sj):.3f} por unidad")

**Preguntas**
9. Interpretá cada peso **en unidades reales**. ¿Cuánto cambian los odds por cada yarda más de distancia? ¿Y por cabecear?
10. Distancia y ángulo están muy correlacionados. ¿Qué efecto puede tener eso sobre los pesos (no sobre las predicciones)?

✍️ **Respuesta:**

## B4. Evaluación

In [ ]:
p_te = xg.predecir_proba(X_te, w, b)
p_sb = op.xg_statsbomb.to_numpy()[test]
base = np.full(test.sum(), y[~test].mean())

tabla = pd.DataFrame({
    "log loss": [xg.log_loss(y[test], q) for q in (base, p_te, p_sb)],
    "AUC": [0.5, xg.auc(y[test], p_te), xg.auc(y[test], p_sb)],
    "xG total": [q.sum() for q in (base, p_te, p_sb)],
}, index=["línea de base", "mi modelo", "StatsBomb"])
print(f"goles reales en test: {y[test].sum()}")
tabla.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for q, nombre in ((p_te, "mi modelo"), (p_sb, "StatsBomb")):
    pm, fr, n = xg.curva_calibracion(y[test], q, n_bins=10)
    ok = n >= 15
    ax.plot(pm[ok], fr[ok], "o-", label=nombre)
    for a, b_, c in zip(pm[ok], fr[ok], n[ok]):
        if nombre == "mi modelo": ax.annotate(str(c), (a, b_), fontsize=7, xytext=(3, -8), textcoords="offset points")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("p predicha (media del intervalo)"); ax.set_ylabel("frecuencia real de gol")
ax.legend(); ax.set_title("calibración (números = tiros en el intervalo)"); plt.show()

**Preguntas**
11. ¿Cuánto mejora tu modelo sobre la línea de base, en log loss? ¿Y cuánto le falta para StatsBomb? Dá dos razones concretas para esa diferencia.
12. ¿Tu modelo está bien calibrado? ¿En qué intervalos **podés** afirmarlo, dada la cantidad de tiros?
13. El "xG total" de tu modelo en test, ¿se parece a los goles reales? ¿Por qué esa comparación es una prueba de calibración global?

✍️ **Respuesta:**

## B5. El mapa de xG

In [ ]:
gx, gy = np.meshgrid(np.linspace(60, 119.5, 240), np.linspace(0, 80, 160))
dg, ag = xg.distancia_y_angulo(gx.ravel(), gy.ravel())
Xg, _, _ = xg.estandarizar(np.column_stack([dg, ag, np.zeros_like(dg)]), m, s)
pg = xg.predecir_proba(Xg, w, b).reshape(gx.shape)

ax = dibujar_media_cancha()
im_ = ax.contourf(gx, gy, pg, levels=[0, .02, .05, .1, .2, .3, .5, .8], cmap="magma", alpha=.85)
plt.colorbar(im_, ax=ax, fraction=.03, label="xG (de pie, jugada)"); dibujar_media_cancha(ax); plt.show()

**Preguntas**
14. ¿Qué forma tienen las curvas de igual xG? ¿Son circunferencias centradas en el arco? Rehacé el mapa con el modelo de **solo distancia** (B2) y compará: ¿qué le aporta el ángulo a la forma, y dónde se nota más?
15. ¿Qué te parece sospechoso del mapa? (Pista: mirá pegado al arco, sobre la línea de fondo, y muy lejos.)

✍️ **Respuesta:**

## B6. *Leakage*: por tiro contra por partido

In [ ]:
# TODO: repetí el entrenamiento y la evaluación con un split POR TIRO (20 % de tiros al azar),
# 20 veces con semillas distintas, y lo mismo con split POR PARTIDO. Compará la media y el desvío
# de la log loss de test en cada caso.

**Preguntas**
16. ¿Hay diferencia? ¿Por qué con **tiros** el *leakage* es chico y con **frames de video** (U6, U7) es enorme?

✍️ **Respuesta:**

## B7. (Opcional) Más información del *freeze frame*

In [ ]:
# TODO (opcional): agregá n_rivales_triangulo y dist_arquero como features.
# Ojo: tienen NaN (tiros sin freeze frame) → decidí qué hacer y justificalo.
# ¿Cuánto mejora la log loss y la AUC? ¿Es una mejora "real" dado el error típico (apunte 05)?

## B8. Hebraica

Completá `unidades/U3-probabilidad/tp3/tiros_hebraica.csv` (ver el enunciado) y corré esto.

In [ ]:
heb = pd.read_csv(RAIZ / "unidades" / "U3-probabilidad" / "tp3" / "tiros_hebraica.csv")
if len(heb) == 0:
    print("Todavía no marcaste tiros: completá tiros_hebraica.csv")
else:
    YD = 1.0936                                           # yardas por metro
    hx = 120 - heb.dist_linea_m.to_numpy() * YD           # a coordenadas StatsBomb
    hy = 40 + heb.lateral_m.to_numpy() * YD
    hd, ha = xg.distancia_y_angulo(hx, hy)
    Xh, _, _ = xg.estandarizar(np.column_stack([hd, ha, (heb.parte_cuerpo == "cabeza").to_numpy()]), m, s)
    heb["xg"] = xg.predecir_proba(Xh, w, b)
    heb.loc[heb.tipo == "Penalty", "xg"] = 0.76           # los penales van aparte (constante)

    ax = dibujar_media_cancha()
    for eq, col in (("hebraica", "tab:blue"), ("rival", "tab:red")):
        h = heb[heb.equipo == eq]
        ax.scatter(120 - h.dist_linea_m * YD, 40 + h.lateral_m * YD, s=600 * h.xg + 20, c=col, alpha=.6,
                   edgecolors=np.where(h.gol == 1, "k", "none"), label=eq)
    ax.legend(); plt.title("tiros (tamaño = xG, borde negro = gol)"); plt.show()
    print(heb.groupby("equipo").agg(tiros=("xg", "size"), xG=("xg", "sum"), goles=("gol", "sum")).round(2))

In [ ]:
# TODO: para cada equipo, con pr.distribucion_goles, calculá la probabilidad de haber hecho
# los goles que hizo (o más) según el modelo. ¿El resultado del partido "fue justo"?

**Preguntas**
17. Según el xG, ¿quién generó más peligro? ¿Coincide con el resultado?
18. ¿Qué podés afirmar sobre si el modelo **sirve** en Sub-21 amateur, con estos datos? ¿Qué **no** podés afirmar? ¿Cuántos tiros harían falta para empezar a medirlo? (Apunte 05, secciones 5 y 6.)

✍️ **Respuesta:**

## Conclusiones

En 5–10 líneas: qué te llevás de U3, qué del modelo de xG usarías en el producto y con qué advertencias, y qué te quedó flojo.

✍️ **Respuesta:**